## Purpose
- Generate realistic asset data for water utility infrastructure
- Create comprehensive asset profiles with operational metrics
- Establish baseline data for resilience scoring and risk modeling

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

print("Libraries imported successfully!")

## Configuration

In [ ]:
# Dataset configuration
N_ASSETS = 8000
CURRENT_YEAR = 2024

# Asset types for water infrastructure
ASSET_TYPES = [
    'Water Treatment Plant', 'Pump Station', 'Distribution Valve', 
    'Storage Tank', 'Pipeline', 'Control System', 'Monitoring Sensor',
    'Filtration Unit', 'Chemical Dosing Unit', 'Flow Meter'
]

# Regional distribution for utility service area
REGIONS = [
    'North Region', 'South Region', 'West Region', 'East Region',
    'Central Region', 'Northeast Region', 'Northwest Region', 'Southeast Region', 'Southwest Region', 'Metro Region'
]

# Redundancy levels
REDUNDANCY_LEVELS = ['None', 'Partial', 'Full', 'N+1', 'N+2']

print(f"Configuration set - generating {N_ASSETS:,} assets")

## Data Generation Functions

In [ ]:
def generate_asset_data(n_assets):
    """
    Generate comprehensive asset data for resilience analysis
    """
    data = []
    
    for i in range(n_assets):
        asset_type = np.random.choice(ASSET_TYPES)
        region = np.random.choice(REGIONS)
        
        # Installation year - skewed towards newer assets
        install_year = np.random.choice(
            range(1980, CURRENT_YEAR + 1),
            p=generate_year_probabilities()
        )
        
        # Age-dependent failure patterns
        asset_age = CURRENT_YEAR - install_year
        age_factor = 1 + (asset_age / 40)  # Higher failure rate for older assets
        
        # Asset type dependent characteristics
        type_multiplier = get_asset_type_multiplier(asset_type)
        
        # Generate correlated operational metrics
        base_downtime = np.random.exponential(50) * age_factor * type_multiplier['downtime']
        downtime_hours = min(max(base_downtime, 0), 8760)  # Cap at hours in a year
        
        # Failure count correlated with downtime
        failure_count = max(0, int(np.random.poisson(downtime_hours / 100) * type_multiplier['failure']))
        
        # MTTR based on asset type and complexity
        base_mttr = np.random.lognormal(2, 0.8) * type_multiplier['mttr']
        mttr = max(1, min(base_mttr, 720))  # Between 1 hour and 30 days
        
        # Maintenance compliance - newer assets tend to have better compliance
        compliance_base = 0.9 - (asset_age * 0.005)  # Decreases with age
        maintenance_compliance = max(0.1, min(1.0, 
            compliance_base + np.random.normal(0, 0.1)))
        
        # Redundancy level based on asset type criticality
        redundancy_weights = get_redundancy_weights(asset_type)
        redundancy_level = np.random.choice(REDUNDANCY_LEVELS, p=redundancy_weights)
        
        # Energy consumption based on asset type
        base_energy = type_multiplier['energy'] * np.random.lognormal(6, 1)
        energy_consumption = max(0, base_energy)
        
        # Criticality score - higher for treatment plants and pump stations
        criticality_base = type_multiplier['criticality']
        criticality_score = max(1, min(10, 
            criticality_base + np.random.normal(0, 1)))
        
        asset_data = {
            'asset_id': f"AST_{i+1:06d}",
            'region': region,
            'asset_type': asset_type,
            'install_year': install_year,
            'downtime_hours_last_year': round(downtime_hours, 2),
            'failure_count_last_year': failure_count,
            'mean_time_to_repair': round(mttr, 2),
            'maintenance_compliance': round(maintenance_compliance, 3),
            'redundancy_level': redundancy_level,
            'energy_consumption_kwh': round(energy_consumption, 2),
            'criticality_score': round(criticality_score, 2)
        }
        
        data.append(asset_data)
    
    return data

def generate_year_probabilities():
    """
    Generate probabilities favoring newer installations
    """
    years = list(range(1980, CURRENT_YEAR + 1))
    # Linear increase in probability towards recent years
    weights = [(year - 1980 + 1) for year in years]
    total_weight = sum(weights)
    return [w/total_weight for w in weights]

def get_asset_type_multiplier(asset_type):
    """
    Return multipliers based on asset type characteristics
    """
    multipliers = {
        'Water Treatment Plant': {
            'downtime': 0.5, 'failure': 0.7, 'mttr': 2.0, 
            'energy': 10.0, 'criticality': 9.0
        },
        'Pump Station': {
            'downtime': 1.0, 'failure': 1.2, 'mttr': 1.5, 
            'energy': 5.0, 'criticality': 8.0
        },
        'Distribution Valve': {
            'downtime': 0.8, 'failure': 0.9, 'mttr': 0.8, 
            'energy': 0.1, 'criticality': 6.0
        },
        'Storage Tank': {
            'downtime': 0.3, 'failure': 0.5, 'mttr': 3.0, 
            'energy': 0.5, 'criticality': 7.0
        },
        'Pipeline': {
            'downtime': 1.5, 'failure': 0.8, 'mttr': 4.0, 
            'energy': 0.0, 'criticality': 5.0
        },
        'Control System': {
            'downtime': 0.4, 'failure': 1.5, 'mttr': 0.5, 
            'energy': 0.8, 'criticality': 8.5
        },
        'Monitoring Sensor': {
            'downtime': 0.2, 'failure': 2.0, 'mttr': 0.3, 
            'energy': 0.1, 'criticality': 4.0
        },
        'Filtration Unit': {
            'downtime': 1.2, 'failure': 1.0, 'mttr': 1.8, 
            'energy': 3.0, 'criticality': 7.5
        },
        'Chemical Dosing Unit': {
            'downtime': 0.6, 'failure': 1.3, 'mttr': 1.0, 
            'energy': 1.0, 'criticality': 6.5
        },
        'Flow Meter': {
            'downtime': 0.3, 'failure': 1.8, 'mttr': 0.4, 
            'energy': 0.2, 'criticality': 5.5
        }
    }
    
    return multipliers.get(asset_type, {
        'downtime': 1.0, 'failure': 1.0, 'mttr': 1.0, 
        'energy': 1.0, 'criticality': 5.0
    })

def get_redundancy_weights(asset_type):
    """
    Return probability weights for redundancy levels based on asset type
    """
    # Critical assets more likely to have redundancy
    if asset_type in ['Water Treatment Plant', 'Pump Station', 'Control System']:
        return [0.1, 0.2, 0.3, 0.3, 0.1]  # Higher redundancy probability
    elif asset_type in ['Storage Tank', 'Filtration Unit']:
        return [0.2, 0.3, 0.3, 0.2, 0.0]  # Moderate redundancy
    else:
        return [0.4, 0.3, 0.2, 0.1, 0.0]  # Lower redundancy for simpler assets

print("Data generation functions defined successfully!")

## Generate Dataset

In [ ]:
print(f"Generating {N_ASSETS:,} asset records...")

# Generate the data
asset_data = generate_asset_data(N_ASSETS)

# Create DataFrame
df = pd.DataFrame(asset_data)

print(f"Dataset generated successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## Data Quality Checks

In [ ]:
# Basic statistics
print("=== DATASET SUMMARY ===")
print(f"Total Assets: {len(df):,}")
print(f"Asset Types: {df['asset_type'].nunique()}")
print(f"Regions: {df['region'].nunique()}")
print(f"Year Range: {df['install_year'].min()} - {df['install_year'].max()}")

print("\n=== ASSET TYPE DISTRIBUTION ===")
print(df['asset_type'].value_counts())

print("\n=== REGIONAL DISTRIBUTION ===")
print(df['region'].value_counts())

print("\n=== REDUNDANCY LEVEL DISTRIBUTION ===")
print(df['redundancy_level'].value_counts())

In [ ]:
# Numerical statistics
print("=== NUMERICAL STATISTICS ===")
numerical_cols = ['downtime_hours_last_year', 'failure_count_last_year', 'mean_time_to_repair', 
                 'maintenance_compliance', 'energy_consumption_kwh', 'criticality_score']

df[numerical_cols].describe()

In [ ]:
# Data quality checks
print("=== DATA QUALITY CHECKS ===")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate asset IDs: {df['asset_id'].duplicated().sum()}")

# Check data ranges
print("\nData Range Validation:")
print(f"Maintenance compliance in [0,1]: {(df['maintenance_compliance'] >= 0).all() and (df['maintenance_compliance'] <= 1).all()}")
print(f"Criticality score in [1,10]: {(df['criticality_score'] >= 1).all() and (df['criticality_score'] <= 10).all()}")
print(f"Non-negative downtime: {(df['downtime_hours_last_year'] >= 0).all()}")
print(f"Non-negative failures: {(df['failure_count_last_year'] >= 0).all()}")
print(f"Positive MTTR: {(df['mean_time_to_repair'] > 0).all()}")

## Save Dataset

In [ ]:
# Create output directory if it doesn't exist
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
output_file = os.path.join(output_dir, 'resilience_assets.csv')
df.to_csv(output_file, index=False)

print(f"Dataset saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

# Verify the saved file
verification_df = pd.read_csv(output_file)
print(f"Verification - Loaded shape: {verification_df.shape}")
print(f"Verification successful: {verification_df.equals(df)}")

## Sample Data Preview

In [ ]:
# Show sample records for each asset type
print("=== SAMPLE RECORDS BY ASSET TYPE ===")
for asset_type in df['asset_type'].unique():
    print(f"\n--- {asset_type} ---")
    sample = df[df['asset_type'] == asset_type].head(2)
    for _, row in sample.iterrows():
        print(f"ID: {row['asset_id']}, Region: {row['region']}, "
              f"Downtime: {row['downtime_hours_last_year']:.1f}h, "
              f"Failures: {row['failure_count_last_year']}, "
              f"Criticality: {row['criticality_score']:.1f}")

## Summary

✅ **Dataset Generation Complete**

- **Assets Generated:** 8,000
- **Asset Types:** 10 categories covering water utility infrastructure
- **Regions:** 10 utility service areas
- **Time Period:** Installation years 1980-2024
- **Output File:** `../data/resilience_assets.csv`

### Key Features:
1. **Realistic Correlations:** Age affects failure rates and maintenance compliance
2. **Asset-Type Specific:** Different failure patterns for pumps vs. sensors vs. treatment plants
3. **Geographic Distribution:** Realistic regional spread across utility service areas
4. **Redundancy Modeling:** Critical assets more likely to have backup systems
5. **Data Quality:** No missing values, proper ranges, unique identifiers

### Next Steps:
- Proceed to `02_resilience_metrics.ipynb` for resilience scoring
- The dataset is ready for machine learning and dashboard visualization